Скачиваем все зависимости

In [ ]:
!pip uninstall -y langchain langchain-core langgraph langgraph-prebuilt langchain-openai langchain-groq

!pip install -q \
langchain==1.0.2 \
langchain-core==1.0.2 \
langgraph==1.0.1 \
langgraph-prebuilt==1.0.1 \
langchain-openai==1.0.1 \
langchain-groq

Found existing installation: langchain 1.0.2
Uninstalling langchain-1.0.2:
  Successfully uninstalled langchain-1.0.2
Found existing installation: langchain-core 1.0.2
Uninstalling langchain-core-1.0.2:
  Successfully uninstalled langchain-core-1.0.2
Found existing installation: langgraph 1.0.1
Uninstalling langgraph-1.0.1:
  Successfully uninstalled langgraph-1.0.1
Found existing installation: langgraph-prebuilt 1.0.1
Uninstalling langgraph-prebuilt-1.0.1:
  Successfully uninstalled langgraph-prebuilt-1.0.1
Found existing installation: langchain-openai 1.0.1
Uninstalling langchain-openai-1.0.1:
  Successfully uninstalled langchain-openai-1.0.1
Found existing installation: langchain-groq 1.0.1
Uninstalling langchain-groq-1.0.1:
  Successfully uninstalled langchain-groq-1.0.1


In [ ]:
!pip install kagglehub -q

In [ ]:
OPENROUTER_TOKEN = ""

Скачивание и обработка датасета

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

os.environ["KAGGLE_TOKEN"] = ""

llm_download = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="anthropic/claude-haiku-4.5",
    temperature=0.1,
    max_tokens=4000,
    model_kwargs={"parallel_tool_calls": False}
)

agent_state = {
    "df_raw": None,
    "df_clean": None,
    "preprocessing_log": [],
}

In [ ]:
# cтолбцы которые не нужны для ML дропаю
DROP_ON_LOAD = [
    "Similar Artist 1", "Similar Song 1", "Similarity Score 1",
    "Similar Artist 2", "Similar Song 2", "Similarity Score 2",
    "Similar Artist 3", "Similar Song 3", "Similarity Score 3",
    "Good for Exercise", "Good for Running", "Good for Yoga/Stretching",
    "Good for Social Gatherings", "Good for Morning Routine",
]


@tool
def download_dataset(dataset_id: str = "devdope/900k-spotify") -> str:
    """Скачивает датасет Spotify с Kaggle. Берёт первые 100к строк.
    Сразу дропает столбцы похожих артистов и ненужные Good for."""
    try:
        import kagglehub
        from pathlib import Path

        print(f"Скачиваю: {dataset_id}...")
        path = kagglehub.dataset_download(dataset_id)
        csv_files = list(Path(path).rglob("*.csv")) #ищем CSV файл в скачанной папке
        csv_file = max(csv_files, key=lambda f: f.stat().st_size)
        df = pd.read_csv(csv_file, low_memory=False) #загружаем весь датасет
        cols_to_drop = [c for c in DROP_ON_LOAD if c in df.columns] #дропаю ненужные столбцы
        df = df.drop(columns=cols_to_drop)
        good_for_cols = ["Good for Party", "Good for Work/Study",
                        "Good for Relaxation/Meditation", "Good for Driving"] #сначала фильтрую по Good for и оставляю где хотя бы 1 из 4 = 1
        good_for_cols = [c for c in good_for_cols if c in df.columns]
        before_filter = len(df)
        df = df[df[good_for_cols].sum(axis=1) >= 1]
        print(f"Good for фильтр: {before_filter:,} → {len(df):,} строк")
        if len(df) > 100_000: #потом берём 100к из отфильтрованных
            df = df.sample(n=100_000, random_state=42)
            print(f"Ограничено до 100k строк")

        output_path = "spotify_original.csv"
        df.to_csv(output_path, index=False, encoding="utf-8-sig")

        agent_state["df_raw"] = df
        return (
            f" датасет загрузили \n"
            f"Строк: {df.shape[0]:,} | Колонок: {df.shape[1]}\n"
            f"Дропнуто: {cols_to_drop}\n\n"
            f"Колонки: {list(df.columns)}\n\n"
            f"Типы:\n{df.dtypes.to_string()}"
        )
    except Exception as e:
        return f"Ошибка: {type(e).__name__}: {e}"

@tool
def analyze_data_quality() -> str:
    """Анализирует качество данных: пропуски, дубликаты,
    статистику числовых колонок, распределение emotion."""
    df = agent_state.get("df_raw")
    if df is None:
        return "Ошибка: сначала вызови download_dataset"
    #считаю пропуски по каждой колонке
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_info = pd.DataFrame({"пропусков": missing, "%": missing_pct})
    missing_info = missing_info[missing_info["пропусков"] > 0].sort_values("%", ascending=False).to_string()
    dups = df.duplicated().sum()
    stats = df.select_dtypes(include=[np.number]).describe().round(2).to_string()
    #смотрию баланс классов emotion
    emotion_info = ""
    if "emotion" in df.columns:
        vc = df["emotion"].value_counts()
        emotion_info = pd.DataFrame({
            "кол-во": vc,
            "%": (vc / len(df) * 100).round(1)
        }).to_string()
    return (
        f" анализ качества \n"
        f"Размер: {df.shape[0]:,} × {df.shape[1]}\n\n"
        f"Пропуски:\n{missing_info or 'нет'}\n\n"
        f"Дубликатов: {dups:,}\n\n"
        f"Числовая статистика:\n{stats}\n\n"
        f"Распределение emotion:\n{emotion_info}"
    )

@tool
def remove_duplicates() -> str:
    """Удаляет полные дубликаты строк и дубликаты
    по паре Artist(s) + song."""
    df = agent_state.get("df_raw")
    if df is None:
        return "Ошибка сначала вызови download_dataset"
    df = df.copy()
    before = len(df)
    #сначала полные дубликаты
    df = df.drop_duplicates()
    removed_full = before - len(df)
    #потом дубликаты по артисту и названию песни
    artist_song_dups = 0
    if "Artist(s)" in df.columns and "song" in df.columns:
        b = len(df)
        df = df.drop_duplicates(subset=["Artist(s)", "song"], keep="first")
        artist_song_dups = b - len(df)
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append(
        f"Дубликаты: удалено {removed_full} полных + {artist_song_dups} по Artist+song"
    )
    return (
        f" дубликаты \n"
        f"До: {before:,} | После: {len(df):,}\n"
        f"Полных удалено: {removed_full:,}\n"
        f"По Artist+song удалено: {artist_song_dups:,}"
    )

@tool
def handle_missing_values() -> str:
    """Заполняет пропуски: строки без emotion удаляет,
    числовые — медианой, категориальные — модой или Unknown."""
    df = agent_state.get("df_clean")
    if df is None:
        df = agent_state.get("df_raw")
    if df is None:
        return "Ошибка: сначала вызови download_dataset"
    df = df.copy()
    log = []
    before = len(df)
    #пришли к выводу что без целевой переменной строка бесполезна для обучения
    if "emotion" in df.columns:
        n = df["emotion"].isnull().sum()
        if n > 0:
            df = df.dropna(subset=["emotion"])
            log.append(f"Удалено {n} строк без emotion")
    #числовые - медиана
    for col in df.select_dtypes(include=[np.number]).columns:
        n = df[col].isnull().sum()
        if n > 0:
            med = df[col].median()
            df[col] = df[col].fillna(med)
            log.append(f"{col}: {n} пропусков → медиана ({med:.2f})")
    #категориальные - мода если мало пропусков, иначе Unknown
    for col in df.select_dtypes(include=["object"]).columns:
        n = df[col].isnull().sum()
        if n == 0:
            continue
        pct = n / len(df) * 100
        if pct < 5:
            mode = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
            df[col] = df[col].fillna(mode)
            log.append(f"{col}: {n} ({pct:.1f}%) → мода '{mode}'")
        else:
            df[col] = df[col].fillna("Unknown")
            log.append(f"{col}: {n} ({pct:.1f}%) → 'Unknown'")
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].extend(log)

    return (
        f" пропуски \n"
        f"Строк до: {before:,} | после: {len(df):,}\n\n"
        + "\n".join(f"  • {l}" for l in log)
        + f"\n\nОсталось пропусков: {df.isnull().sum().sum()}"
    )


@tool
def convert_length_to_seconds() -> str:
    """Конвертирует столбец Length из формата MM:SS в секунды."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None: return "Ошибка: нет данных"
    if "Length" not in df.columns: return "Столбец Length не найден"
    df = df.copy()
    def parse_len(v):
        try:
            p = str(v).strip().split(":")
            return int(p[0]) * 60 + int(p[1]) if len(p) == 2 else np.nan
        except: return np.nan
    df["length_seconds"] = df["Length"].apply(parse_len)
    df["length_seconds"] = df["length_seconds"].fillna(df["length_seconds"].median()).astype(int)
    df = df.drop(columns=["Length"])
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Length → length_seconds (секунды)")
    return f"Готово: Length → length_seconds. Пример: {df['length_seconds'].head(3).tolist()}"


@tool
def convert_release_date() -> str:
    """Конвертирует Release Date в числовые столбцы release_year и release_month."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None: return "Ошибка: нет данных"
    if "Release Date" not in df.columns: return "Столбец Release Date не найден"
    df = df.copy()
    def parse_date(v):
        try: return pd.to_datetime(str(v), dayfirst=True)
        except: return pd.NaT
    dates = df["Release Date"].apply(parse_date)
    df["release_year"] = dates.dt.year.fillna(2000).astype(int)
    df["release_month"] = dates.dt.month.fillna(6).astype(int)
    df = df.drop(columns=["Release Date"])
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Release Date → release_year + release_month")
    return f"Готово: Release Date → release_year + release_month. Пример года: {df['release_year'].head(3).tolist()}"


@tool
def convert_loudness() -> str:
    """Конвертирует Loudness (db) из строки в числовой float."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None: return "Ошибка: нет данных"
    if "Loudness (db)" not in df.columns: return "Столбец Loudness (db) не найден"
    df = df.copy()
    df["loudness_db"] = pd.to_numeric(
        df["Loudness (db)"].astype(str).str.replace("[^0-9\\-\\.]", "", regex=True),
        errors="coerce"
    )
    df["loudness_db"] = df["loudness_db"].fillna(df["loudness_db"].median())
    df = df.drop(columns=["Loudness (db)"])
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Loudness (db) → loudness_db (float)")
    return f"Готово: Loudness (db) → loudness_db. Пример: {df['loudness_db'].head(3).tolist()}"


@tool
def convert_explicit() -> str:
    """Конвертирует Explicit из True/False в бинарный 0/1."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None: return "Ошибка: нет данных"
    if "Explicit" not in df.columns: return "Столбец Explicit не найден"
    df = df.copy()
    df["explicit_int"] = df["Explicit"].astype(str).str.lower().map(
        lambda x: 1 if x in ["true", "1", "yes", "explicit"] else 0
    )
    df = df.drop(columns=["Explicit"])
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Explicit → explicit_int (0/1)")
    return f"Готово: Explicit → explicit_int. Распределение: {df['explicit_int'].value_counts().to_dict()}"


@tool
def convert_key() -> str:
    """Конвертирует Key из формата 'A min' в key_note (нота) и key_mode (0=min, 1=Maj)."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None: return "Ошибка: нет данных"
    if "Key" not in df.columns: return "Столбец Key не найден"
    df = df.copy()
    df["key_note"] = df["Key"].astype(str).str.split().str[0]
    df["key_mode"] = df["Key"].astype(str).str.split().str[-1].map(
        {"Maj": 1, "min": 0}
    ).fillna(0).astype(int)
    df = df.drop(columns=["Key"])
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Key → key_note + key_mode (0=min, 1=Maj)")
    return f"Готово: Key → key_note + key_mode. Примеры нот: {df['key_note'].value_counts().head(3).to_dict()}"


@tool
def convert_time_signature() -> str:
    """Конвертирует Time signature из '4/4' в числовой int (числитель)."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None: return "Ошибка: нет данных"
    if "Time signature" not in df.columns: return "Столбец Time signature не найден"
    df = df.copy()
    df["time_sig"] = pd.to_numeric(
        df["Time signature"].astype(str).str.split("/").str[0],
        errors="coerce"
    ).fillna(4).astype(int)
    df = df.drop(columns=["Time signature"])
    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Time signature → time_sig (int)")
    return f"Готово: Time signature → time_sig. Распределение: {df['time_sig'].value_counts().to_dict()}"


@tool
def handle_outliers() -> str:
    """Обрезает выбросы методом IQR. Не удаляет строки —
    заменяет аномальные значения на граничные (capping)."""
    df = agent_state.get("df_clean")
    if df is None:
        return "Ошибка: сначала вызови handle_missing_values"

    df = df.copy()
    log = []

    #проверяю только числовые признаки модели
    cols = ["Tempo", "Energy", "Danceability", "Positiveness", "Speechiness",
            "Liveness", "Acousticness", "Instrumentalness", "Popularity",
            "length_seconds", "loudness_db"]
    cols = [c for c in cols if c in df.columns]

    total = 0
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lo = Q1 - 1.5 * IQR
        hi = Q3 + 1.5 * IQR
        n = ((df[col] < lo) | (df[col] > hi)).sum()
        if n > 0:
            df[col] = df[col].clip(lo, hi)  #clip вместо удаления строк
            log.append(f"{col}: {n:,} выбросов → clip [{lo:.2f}, {hi:.2f}]")
            total += n

    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].extend(log)

    return (
        f" выбросы (IQR)\n"
        f"Проверено колонок: {len(cols)}\n"
        f"Всего значений обрезано: {total:,}\n\n"
        + ("\n".join(f"  • {l}" for l in log) or "Выбросов не найдено")
    )

@tool
def save_clean_dataset_and_report() -> str:
    """Сохраняет чистый датасет в spotify_clean.csv
    и выводит финальный отчёт о всех шагах предобработки."""
    df_raw = agent_state.get("df_raw")
    df_clean = agent_state.get("df_clean")
    log = agent_state.get("preprocessing_log", [])

    if df_clean is None:
        return "Ошибка: нет очищенных данных"

    df_clean.to_csv("spotify_clean.csv", index=False)

    report = (
        f"финальный отчет\n"
        f"Было:  {len(df_raw):,} строк × {df_raw.shape[1]} колонок\n"
        f"Стало: {len(df_clean):,} строк × {df_clean.shape[1]} колонок\n\n"
        f"Шаги предобработки:\n"
        + "\n".join(f"  {i+1}. {s}" for i, s in enumerate(log))
        + f"\n\nИтоговые колонки:\n{list(df_clean.columns)}\n\n"
        f"Пропусков: {df_clean.isnull().sum().sum()}\n"
        f"Дубликатов: {df_clean.duplicated().sum()}\n\n"
        f"Файл сохранён: spotify_clean.csv"
    )
    print(report)
    return report

@tool
def filter_emotions() -> str:
    """Оставляет только валидные классы emotion:
    joy, sadness, anger, fear, love, surprise.
    Убирает мусорные значения типа True, angry и т.д."""
    df = agent_state.get("df_clean")
    if df is None:
        return "Ошибка: сначала вызови fix_data_types"

    df = df.copy()
    before = len(df)

    #только эти 6 классов валидны
    valid_emotions = ["joy", "sadness", "anger", "fear", "love", "surprise"]
    df = df[df["emotion"].isin(valid_emotions)]

    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append(
        f"emotion: оставлены только {valid_emotions}, удалено {before - len(df)} мусорных строк"
    )
    return (
        f"=== ФИЛЬТРАЦИЯ EMOTION ===\n"
        f"До: {before:,} | После: {len(df):,}\n"
        f"Удалено мусорных строк: {before - len(df)}\n\n"
        f"Распределение emotion:\n"
        + df["emotion"].value_counts().to_string()
    )

@tool
def analyze_columns_and_decide() -> str:
    """Показывает агенту примеры данных из каждого столбца.
    Агент сам анализирует содержимое и решает какие конвертеры применить."""
    df = agent_state.get("df_clean")
    if df is None:
      df = agent_state.get("df_raw")
    if df is None:
        return "Ошибка: нет данных"

    report = "=== ПРИМЕРЫ ДАННЫХ ПО КАЖДОМУ СТОЛБЦУ ===\n\n"
    report += "Проанализируй каждый столбец и реши какие преобразования нужны.\n\n"

    for col in df.columns:
        dtype = str(df[col].dtype)
        sample = df[col].dropna().head(3).tolist()
        n_unique = df[col].nunique()
        report += f"• '{col}' | тип: {dtype} | уникальных: {n_unique}\n"
        report += f"  примеры: {sample}\n\n"

    report += (
        "\nДоступные конвертеры:\n"
        "- convert_length_to_seconds\n"
        "- convert_release_date\n"
        "- convert_loudness\n"
        "- convert_explicit\n"
        "- convert_key\n"
        "- convert_time_signature\n\n"
        "Реши самостоятельно какие из них нужно вызвать и вызови их."
    )

    return report

@tool
def group_genres() -> str:
    """Разбивает Genre на 7 бинарных столбцов жанров.
    Один трек может относиться к нескольким категориям."""
    df = agent_state.get("df_clean")
    if df is None:
        return "Ошибка: сначала вызови filter_emotions"

    df = df.copy()

    genre_groups = {
        "pop": "pop", "k-pop": "pop", "j-pop": "pop",
        "synthpop": "pop", "electropop": "pop",
        "indie pop": "pop", "pop rock": "pop", "pop punk": "pop",
        "rock": "rock", "indie rock": "rock", "alternative rock": "rock",
        "classic rock": "rock", "hard rock": "rock", "garage rock": "rock",
        "britpop": "rock", "post-punk": "rock", "grunge": "rock",
        "math rock": "rock", "shoegaze": "rock", "psychedelic rock": "rock",
        "progressive rock": "rock", "punk": "rock", "punk rock": "rock",
        "emo": "rock", "screamo": "rock", "indie": "rock", "alternative": "rock",
        "metal": "metal", "heavy metal": "metal", "black metal": "metal",
        "death metal": "metal", "doom metal": "metal", "thrash metal": "metal",
        "power metal": "metal", "metalcore": "metal", "deathcore": "metal",
        "nu metal": "metal", "progressive metal": "metal",
        "hip hop": "hiphop_rap", "hip-hop": "hiphop_rap", "rap": "hiphop_rap",
        "trap": "hiphop_rap", "cloud rap": "hiphop_rap",
        "emo rap": "hiphop_rap", "grime": "hiphop_rap",
        "electronic": "electronic_dance", "electro": "electronic_dance",
        "techno": "electronic_dance", "house": "electronic_dance",
        "trance": "electronic_dance", "dubstep": "electronic_dance",
        "drum and bass": "electronic_dance", "dance": "electronic_dance",
        "disco": "electronic_dance", "chillout": "electronic_dance",
        "ambient": "electronic_dance", "lo-fi": "electronic_dance",
        "trip-hop": "electronic_dance", "dream pop": "electronic_dance",
        "jazz": "jazz_blues_soul_funk", "blues": "jazz_blues_soul_funk",
        "soul": "jazz_blues_soul_funk", "funk": "jazz_blues_soul_funk",
        "rnb": "jazz_blues_soul_funk", "swing": "jazz_blues_soul_funk",
        "reggae": "jazz_blues_soul_funk", "latin": "jazz_blues_soul_funk",
        "reggaeton": "jazz_blues_soul_funk",
        "classical": "classical_soundtrack", "soundtrack": "classical_soundtrack",
        "acoustic": "classical_soundtrack", "folk": "classical_soundtrack",
        "country": "classical_soundtrack",
    }

    groups = ["pop", "rock", "metal", "hiphop_rap",
              "electronic_dance", "jazz_blues_soul_funk", "classical_soundtrack"]

    def encode_row(genre_str):
        result = {g: 0 for g in groups}
        if pd.isna(genre_str):
            return pd.Series(result)
        genres = [g.strip().lower() for g in str(genre_str).split(",")]
        for g in genres:
            if g in genre_groups:
                result[genre_groups[g]] = 1
        return pd.Series(result)

    df[groups] = df["Genre"].apply(encode_row)

    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append("Genre → 7 бинарных столбцов жанров")

    return (
        f"жанры\n"
        f"Добавлены столбцы: {groups}\n\n"
        f"Треков по категориям:\n"
        + df[groups].sum().sort_values(ascending=False).to_string()
    )
@tool
def analyze_for_feature_engineering() -> str:
    """Показывает числовые колонки для анализа.
    Агент смотрит на данные и решает какие признаки создать."""
    df = agent_state.get("df_clean")
    if df is None:
        return "Ошибка: нет данных"

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    report = "=== АНАЛИЗ ДЛЯ FEATURE ENGINEERING ===\n\n"
    report += f"Числовые колонки: {numeric_cols}\n\n"
    report += df[numeric_cols].describe().round(2).to_string()
    report += (
        "\n\nДоступные операции:\n"
        "- create_features — создаёт новые признаки\n\n"
        "Реши какие признаки полезны для предсказания emotion и вызови create_features."
    )
    return report


@tool
def create_features() -> str:
    """Создаёт новые признаки на основе анализа данных агентом."""
    df = agent_state.get("df_clean")
    if df is None:
        return "Ошибка: нет данных"

    df = df.copy()
    log = []

    if "Energy" in df.columns and "Acousticness" in df.columns:
        df["energy_acoustic_ratio"] = (df["Energy"] / (df["Acousticness"] + 1)).round(4)
        log.append("energy_acoustic_ratio = Energy / (Acousticness + 1)")

    if "Danceability" in df.columns and "Positiveness" in df.columns:
        df["dance_mood_index"] = (df["Danceability"] * df["Positiveness"] / 100).round(4)
        log.append("dance_mood_index = Danceability * Positiveness / 100")

    if "release_year" in df.columns:
        df["track_age"] = (2025 - df["release_year"]).clip(0, 100)
        log.append("track_age = 2025 - release_year")

    if "Speechiness" in df.columns:
        df["is_speech_heavy"] = (df["Speechiness"] > 33).astype(int)
        log.append("is_speech_heavy = Speechiness > 33")

    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append(f"Feature Engineering: {len(log)} новых признаков")

    return (
        f"FEATURE ENGINEERING\n"
        f"Создано признаков: {len(log)}\n\n"
        + "\n".join(f"  {i+1}. {l}" for i, l in enumerate(log))
    )
@tool
def drop_original_columns() -> str:
    """Удаляет оригинальные столбцы которые уже были сконвертированы
    в новые числовые признаки и больше не нужны."""
    df = agent_state.get("df_clean")
    if df is None:
        return "Ошибка: нет данных"

    df = df.copy()
    # Оригиналы которые уже заменены конвертированными версиями
    cols_to_drop = ["Length", "Key", "Loudness (db)",
                    "Time signature", "Explicit", "Release Date"]
    dropped = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=dropped)

    agent_state["df_clean"] = df
    agent_state["preprocessing_log"].append(
        f"Дропнуты оригинальные столбцы после конвертации: {dropped}"
    )

    return (
        f"=== УДАЛЕНИЕ ОРИГИНАЛЬНЫХ СТОЛБЦОВ ===\n"
        f"Дропнуто: {dropped}\n"
        f"Итого колонок: {df.shape[1]}\n"
        f"Колонки: {list(df.columns)}"
    )

In [ ]:
tools_download = [
    download_dataset,
    analyze_data_quality,
    remove_duplicates,
    handle_missing_values,
    drop_original_columns,
    analyze_columns_and_decide,
    convert_length_to_seconds,
    convert_release_date,
    convert_loudness,
    convert_explicit,
    convert_key,
    convert_time_signature,
    filter_emotions,
    group_genres,
    handle_outliers,
    analyze_for_feature_engineering,
    create_features,
    save_clean_dataset_and_report,
]

agent_download = create_agent(
    model=llm_download,
    tools=tools_download,
    system_prompt=(
        "Ты опытный Data Engineer в музыкальном стриминговом сервисе. "
        "Твоя задача — подготовить датасет Spotify для ML задачи предсказания эмоции трека.\n\n"
        "Порядок работы:\n"
        "1. download_dataset — загрузи данные\n"
        "2. analyze_data_quality — проанализируй качество\n"
        "3. remove_duplicates — удали дубликаты\n"
        "4. handle_missing_values — заполни пропуски\n"
        "5. analyze_columns_and_decide — изучи примеры данных и сам реши какие конвертеры вызвать\n"
        "6. вызови нужные конвертеры самостоятельно\n"
        "7. filter_emotions — оставь только валидные классы\n"
        "8. group_genres — разбей жанры на бинарные столбцы\n"
        "9. handle_outliers — обработай выбросы\n"
        "10. drop_original_columns — удали оригинальные столбцы после конвертации\n"
        "11. analyze_for_feature_engineering — изучи данные\n"
        "12. create_features — создай новые признаки\n"
        "13. save_clean_dataset_and_report — сохрани результат\n\n"
        "Вызывай инструменты строго по одному. Никогда не вызывай несколько tools одновременно."
    )
)

In [ ]:
from langchain_core.messages import AIMessage

for answer in agent_download.stream(
    {"messages": [{"role": "user", "content": "Выполни полный пайплайн подготовки данных по всем 13 шагам. После каждого шага объясняй что сделал и почему."}]},
    config={"recursion_limit": 80},
    stream_mode="updates"
):
    print(answer)
    print()
    for step, info in answer.items():
        message = info["messages"][-1]
        if isinstance(message, AIMessage):
            print(message.content)

{'model': {'messages': [AIMessage(content='Отлично! Начинаю полный пайплайн подготовки данных. Буду выполнять шаги строго по одному и объяснять каждый.\n\n## Шаг 1: Загрузка данных', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 5823, 'total_tokens': 5915, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.006283, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.006283, 'upstream_inference_prompt_cost': 0.005823, 'upstream_inference_completions_cost': 0.00046}}, 'model_provider': 'openai', 'model_name': 'anthropic/claude-4.5-haiku-20251001', 'system_fingerprint': None, 'id': 'gen-1777328508-OFGL0HmSr3tGBLKCIVTJ', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--4

EDA датасета

In [ ]:
from langchain_openai import ChatOpenAI

llm_eda = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="anthropic/claude-haiku-4.5"
)

In [ ]:
from langchain_core.tools import tool
import pandas as pd


@tool
def dataset_summary(path: str, sample_size: int = 2):
    """Тул выводит описание основых характеристик и пример нескольких строк из датасета"""
    df = pd.read_csv(path)

    sample = df.head(sample_size).to_dict(orient="records")
    summary = {
        "shape": df.shape,
        "columns": df.columns.tolist(),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "missing": df.isnull().sum().to_dict(),
        "sample": sample
    }

    return str(summary)


@tool
def dataset_statistics(path: str):
    """Тул выводит статистические характеристики датасета"""
    df = pd.read_csv(path)

    return df.describe(include="all").to_string()

In [ ]:
tools_eda = [dataset_summary, dataset_statistics]

In [ ]:
from langchain.agents import create_agent

agent_eda = create_agent(
    model=llm_eda,
    tools=tools_eda,
    system_prompt="""
    Ты ReAct агент аналитик для EDA анализа датасетов. Твоя задача — НЕ просто вывести данные, а сделать осмысленный анализ на основе фактов.

    Порядок действий:
    1. Всегда сначала вызови dataset_summary для получения основной информации о датасете
    2. Затем dataset_statistics для получения информации о статистических характеристиках датасета

    После этого обязательно сделай аналитический вывод.



    ФИНАЛЬНЫЙ ОТВЕТ ДОЛЖЕН ОБЯЗАТЕЛЬНО БЫТЬ В ФОРМАТЕ:

    Основное описание датасета:
    - Shape:
    - Columns:
    - Data types:
    - Missing values

    Предметная область датасета:
    - Что это за данные, какая предметная область
    - Что представляет одна строка
    - Что значат столбцы

    Ключевые статистические наблюдения:
    - 2-4 наблюдения по данным (распределения, выбросы, странности)

    Потенциальные проблемы:
    - Пропуски
    - Дисбаланс
    - Ненужные признаки

    Идеи для обучения ML моделей:
    - 2-3 идеи, что можно предсказывать



    ЖЕСТКИЕ ПРАВИЛА:
    - Не копируй сырые данные в финальный ответ
    - Не перечисляй всё подряд
    - Делай выводы, а не просто описание
    - Пиши кратко и по делу
    - Обязательно пиши ответ на русском языке
    """
)

In [ ]:
result = agent_eda.invoke({
    "messages": [
        ("user", "Проанализируй датасет по пути test.csv")
    ]
})

print(result["messages"][-1].content)

**Основное описание датасета:**
- Shape: 30 строк, 39 столбцов.
- Columns: включают метаданные трека (исполнитель, название, текст песни, длительность, эмоция, жанр, альбом, дата выпуска, ключ), аудио-характеристики (темп, громкость, размер, эксплицитность, популярность, энергетика, танцевальность, позитивность, речевость, живость, акустичность, инструменталность), бинарные флаги «подходит для» (вечеринка, работа/учёба, релаксация, упражнения, бег, йога, вождение, социальные собрания, утренний ритуал) и три набора схожих исполнителей/песен с оценками сходства.
- Data types: смешанные – объектные (строки) для категориальных и текстовых полей, int64 для числовых признаков и бинарных флагов, float64 для оценок схожести.
- Missing values: отсутствуют во всех столбцах.

**Предметная область датасета:**
- Данные описывают музыкальные треки с их lyrical content и аудио-features, вероятно собранные для задач рекомендации, классификации настроения или прогнозирования популярности.
- Одна строка

Добавление разметки от ЛЛМ

In [ ]:
import re
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm_razm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="",
    model="anthropic/claude-haiku-4.5",
    temperature=0.1, #тут креативность, по базе берет вроде 0,7 а нам нужны более точные ответы
    max_tokens=4000
)

response = llm_razm.invoke("Привет, как дела?")
print(response.content)

Привет! 👋 Спасибо за вопрос! У меня всё хорошо. Я готов помочь — чем я могу тебе помочь?


In [ ]:
#поясняю почти все, чтобы все разобрались
feature_selection = None

playlists = ["party", "work", "relax", "drive"]

category_cols = ["Good for Party","Good for Work/Study","Good for Relaxation/Meditation","Good for Driving"]

key_cols = ['Artist(s)', 'song', 'text', 'emotion', 'Genre', 'Album']

@tool
def prepare_dataset_tool(input_path, output_filename="spotify_for_agent.csv"):
    """Подготавливает датасет для агента."""

    input_path = Path(input_path) #строку с путем в обьект
    output_path = input_path.parent / output_filename #/склеиваем путь

    df_full = pd.read_csv(input_path)

    if len(df_full) > 5500:
      df_full = df_full.sample(n=5500, random_state=42)

    df_result = df_full.drop(columns=category_cols)
    df_result[playlists] = 0
    df_result.to_csv(output_path, index=False)

    return json.dumps({"output": str(output_path)})

@tool
def select_features_tool(input_path: str):
    """
    Выбирает признаки датасета, полезные для классификации песен по плейлистам.
    Работает с датасетом через путь к файлу.
    """

    global feature_selection

    input_path = Path(input_path)
    df_result = pd.read_csv(input_path)

    sample = df_result.sample(5, random_state=42).to_dict("records") #records, чтобы каждая строка была как отдельный словарь

    prompt = f"""You are a music analysis expert.

Playlists: {playlists}

All dataset columns:
{df_result.columns.tolist()}

Examples, 5 random rows:
{json.dumps(sample, default=str)[:2000]}

For each playlist, select the relevant features for classifying songs.
Use exact column names from the dataset.

Return only valid JSON:

{{"feature_selection": {{
    "party": {{"selected_features": [...]}},
    "work":  {{"selected_features": [...]}},
    "relax": {{"selected_features": [...]}},
    "drive": {{"selected_features": [...]}}
}}}}"""

    for attempt in range(3):
        try:
            content = llm_razm.invoke(prompt).content
            content = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', content) #убираю эти невидимые символы, из-за этого модель ложилась часто
            match = re.search(r'\{[\s\S]*\}', content) #  все содержимое между {  }

            if match is None:
                raise ValueError("клод не вернул джисон")

            parsed = json.loads(match.group(0)) #снова текст в словарь
            feature_selection = parsed["feature_selection"]

            return json.dumps(parsed)

        except Exception as e: #на какой  попытке ошибка и что за ошибка
            print(f"Попытка {attempt + 1}: {e}")

    raise ValueError("клод не смог вернуть джисон за 3 попытки")


@tool
def classify_songs_tool(input_path: str, output_filename: str = "classified_songs.csv", n_rows: int = 5500, random_seed: int = 42):
    """Классифицирует n_rows случайных песен по плейлистам. Загружает датасет по пути и сохраняет новый файл."""

    global feature_selection

    if feature_selection is None:
        raise ValueError("Сначала нужно вызвать select_features_tool")

    input_path = Path(input_path)
    output_path = input_path.parent / output_filename

    df_result = pd.read_csv(input_path)

    sample_df = df_result.sample(n_rows, random_state=random_seed).reset_index(drop=False)

    selected = set()
    for playlist_name in playlists:
        selected.update(feature_selection[playlist_name]["selected_features"])

    selected = list(set(selected) & set(df_result.columns))

    template_parts = []
    for playlist_name in playlists:
        part = f'"{playlist_name}": 0'
        template_parts.append(part)

    template = ", ".join(template_parts) #обькдиняем в строку

    batch_size = 15
    batches = []

    for i in range(0, len(sample_df), batch_size):
        batch = sample_df.iloc[i:i + batch_size]
        batches.append(batch)

    for batch_idx, batch_df in enumerate(tqdm(batches, desc="Batches")):

        if batch_idx > 0 and batch_idx % 50 == 0:
            checkpoint_path = input_path.parent / "checkpoint.csv"
            df_result.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

        songs = batch_df[selected].to_dict("records") #беру ток нужные
        original_indices = batch_df["index"].tolist()

        prompt = f"""Classify each of {len(songs)} songs into 4 independent playlists: {playlists}.

For each playlist output 1 if the song fits and 0 if it does not fit.

Songs:
{json.dumps(songs, default=str)}

Return only valid JSON:

{{"results": [{{"song_index": 0, "playlist_binary": {{{template}}}}}]}}

Output exactly {len(songs)} objects in the same order.
"""

        try:
            content = llm_razm.invoke(prompt).content
            match = re.search(r'\{[\s\S]*\}', content)

            if match is None: #тут ошибку делаю, если вернула не джисон
                raise ValueError("клод бернул не джисон")

            parsed = json.loads(match.group(0))

            for song_result in parsed["results"]:
                position_in_batch = song_result["song_index"] #определяет позицию внутри батча, чтобы вернуть на то же место

                if position_in_batch >= len(original_indices): #когда тупая иишка, она может вернуть больше песен чем в батче
                    continue

                row_index = original_indices[position_in_batch] #позицию в батче переводит в реальный индекс
                playlist_binary = song_result["playlist_binary"]

                for playlist_name in playlists:
                    value = int(playlist_binary.get(playlist_name, 0))
                    df_result.at[row_index, playlist_name] = value

        except Exception as e:
            print(f"Batch {batch_idx} skipped: {e}")
            continue

    df_result.to_csv(output_path, index=False, encoding="utf-8-sig")

    summary = {playlist_name: int(df_result[playlist_name].sum()) for playlist_name in playlists}

    return json.dumps({"status": "ok","saved_to": str(output_path),"rows": len(df_result)})


@tool
def merge_extra_columns_tool(original_path, classified_path, output_filename="df_final.csv"):
    """Добавляет обратно колонки Good for в итоговый датасет."""

    df_original = pd.read_csv(original_path)
    df_classified = pd.read_csv(classified_path)
    df_extra = df_original[key_cols + category_cols]

    df_final = pd.merge(df_classified, df_extra, on=key_cols, how="left")

    df_final.to_csv(output_filename, index=False)

    return json.dumps({"saved_to": output_filename})

tools_razm = [prepare_dataset_tool,select_features_tool,classify_songs_tool,merge_extra_columns_tool]

In [ ]:
from langchain.agents import create_agent

agent_razm = create_agent(
    model=llm_razm,
    tools=tools_razm,
    system_prompt=(
        """Ты ИИ агент для классификации музыки по 4 плейлистам: party, work, relax, drive

КРИТИЧЕСКИ ВАЖНО:
Работай с датасетом только через путь к файлу
Не используй глобальные переменные датафреймов
Все преобразования датасета делай только через tools
Вызывай tools по одному, никогда не несколько одновременно
Не повторяй вызов одного и того же tool, если он уже успешно сработал

Порядок работы:
Шаг 1: prepare_dataset_tool - подготовь датасет из исходного файла
Шаг 2: select_features_tool — выбери признаки по подготовленному файлу
Шаг 3: classify_songs_tool — классифицируй песни и сохрани classified_songs.csv
Шаг 4: merge_extra_columns_tool — верни колонки Good for из исходного файла и сохрани df_final.csv
Шаг 5: дай финальный ответ — где лежит итоговый файл и сколько песен в каждом плейлисте

После merge_extra_columns_tool остановись Не вызывай больше tools"""))

In [ ]:
input_path = "spotify_clean.csv"

result = agent_razm.invoke({
    "messages": [{
        "role": "user",
        "content": f"""
У тебя есть исходный датасет по пути: {input_path}.

Нужно:
1. Подготовить датасет для агента: убрать колонки Good for..., потому что по ним нельзя классифицировать.
2. Сохранить подготовленный файл рядом с исходным датасетом.
3. Выбрать признаки для классификации.
4. Классифицировать 5500 случайных песен по 4 плейлистам: party, work, relax, drive.
5. Сохранить результат в classified_songs.csv.
6. Вернуть в финальный датасет колонки Good for из исходного файла.
7. Сохранить финальный файл как df_final.csv.
""" }] })

print(result["messages"][-1].content)

Batches:   4%|█▎                               | 14/367 [00:45<19:20,  3.29s/it]

In [ ]:
llm_preproc = ChatOpenAI(api_key = OPENROUTER_TOKEN, base_url = 'https://openrouter.ai/api/v1', model = 'anthropic/claude-haiku-4.5')

In [ ]:
import json

@tool
def analyze_columns(file_path: str) -> str:
    """ Тул для анализа датасета. Проходим по каждой колонке и записываем про нее информацию о:
        - типе данных
        - количестве уникальных значений, которые принимает признак """
    df = pd.read_csv(file_path)

    information = {}
    columns = df.columns
    for column in columns:
        information[column] = {'column_type': str(df[column].dtype), 'column_unique_names': df[column].nunique()}

    with open('analysis.json', 'w', encoding = 'utf-8') as f:
        json.dump(information, f, ensure_ascii = False)
    return 'analysis.json'

@tool
def decision_column(information_path: str, number: int) -> str:
    """ Тул для разделения колонок с типом данных object на те, которые далее подлежат удалению и те, которые далее будут закодированы"""
    with open(information_path, "r", encoding="utf-8") as f:
        information = json.load(f)

    drop_columns = []
    ohe_columns = []
    for column in information:
        column_dtype = information[column]['column_type']
        column_nunique = information[column]['column_unique_names']
        if column_dtype == 'object':
            if column_nunique < number:
                ohe_columns.append(column)
            else:
                drop_columns.append(column)

    with open('decision.json', 'w', encoding = 'utf-8') as f:
        json.dump({'drop_columns': drop_columns, 'ohe_columns': ohe_columns}, f, ensure_ascii = False)
    return 'decision.json'

@tool
def work_with_columns(file_path: str, decision_path: str) -> str:
    """ Тул для подготовки датасета для дальнейшей ML задачи.
        Удаляем ненужные object столбцы, а остальные столбцы object кодируем с помощью one-hot-encoding """
    df = pd.read_csv(file_path)

    with open(decision_path, 'r', encoding = 'utf-8') as f:
        decision = json.load(f)

    # столбцы для удаления
    drop_columns = decision['drop_columns']
    df = df.drop(columns = drop_columns)

    # столбцы для кодировки
    ohe_columns = decision['ohe_columns']
    df = pd.get_dummies(df, columns = ohe_columns)

    output_path = 'cleaned_dataset.csv'
    df.to_csv(output_path, index = False)
    return output_path


tools_preproc = [analyze_columns, decision_column, work_with_columns]

In [ ]:
from langchain.agents import create_agent

agent_preproc = create_agent(
    model = llm_preproc,
    tools = tools_preproc,
    system_prompt = """
    Ты агент для подготовки табличных данных для задач машинного обучения.
    Твоя цель - превратить сырой датасет в готовый набор признаков для модели, закодировав нужные object столбцы и удалив ненужные object
    столбцы, числовые не трогай.
    Ты должен работать только через tools и тебе нельзя выполнять преобразования вручную.

    Шаги твой работы:
    Шаг 1. Сначала проанализируй датасет через tool анализа колонок (analyze_columns):
       Ты получишь информацию о:
       - типах данных
       - количестве уникальных значений
       Важные правила:
       - инструмент analyze_columns ты можешь вызывать только один раз
       - если анализ уже выполнен, не вызывай инструмен analyze_columns повторно
       - используй результат анализа как окончательный

    Шаг 2. Определи допустимое количество уникальных значений в колрнке для OHE:
       Порог уникальности (допустимое количество уникальных значений для кодировки (number)) подбирай сам,
       но выбирай обычно в диапазоне 20–50, в зависимости от данных

    Шаг 3. Передай результат в tool принятия решения (decision_column):
       - обязательно передавай в tool под information_path значение 'analysis.json' и выбранный тобой порог уникальности (number)
       - какие колонки удалить
       - какие закодировать one-hot encoding

    Шаг 4. Используй tool для преобразования датасета (work_with_columns), чтобы:
       - удалить выбранные колонки
       - применить one-hot encoding
       - сохранить итоговый датасет

    Обязательно в шаге 4 передавай в tool под decision_path значение 'decision.json'

    Ключевые правила:
    - всегда начинай с analyze_columns
    - все tools вызываются только один раз за задачу
    - не повторяй уже выполненные шаги
    - не делай повторный анализ данных
    - используй только информацию из tools
    - не придумывай значения и не делай предположений без анализа
    - сам выбирай порог уникальности в зависимости от структуры данных
    - финальный результат — это очищенный и закодированный датасет

    Самопроверка перед каждым вызовом инструмента:

    Перед вызовом следующего tool проверь:
    1. Выполнен ли предыдущий шаг пайплайна?
    2. Не вызывался ли этот tool уже ранее?
    3. Корректные ли аргументы будут переданы в tool?
    4. Не завершена ли задача уже (если создан cleaned_dataset.csv — остановись).

    Если хотя бы один ответ отрицательный — не выполняй новый вызов tool, сначала исправь последовательность действий.
    """)

In [ ]:
result = agent_preproc.invoke({
    "messages": [
        ("user", "Очисти и закодируй датасет: 'df_final.csv'")
    ]
})

print(result["messages"][-1].content)

✅ **Задача завершена успешно!**

Датасет полностью подготовлен и сохранен в файл **`cleaned_dataset.csv`**

**Что было сделано:**
1. ✓ Проанализирован датасет и определены типы данных и количество уникальных значений
2. ✓ Установлен порог уникальности в 25 значений для one-hot encoding
3. ✓ Определены колонки для удаления и кодирования
4. ✓ Удалены ненужные object колонки
5. ✓ Применено one-hot encoding к оставшимся категориальным признакам
6. ✓ Числовые колонки остались без изменений

Финальный датасет готов к использованию в ML моделях! 🚀


Делаем обучение ML моделей

In [ ]:
!pip install lightgbm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingClassifier
)

from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from lightgbm import LGBMClassifier, LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
test_size = 0.2
val_size = 0.25
random_state = 42

task_type = None
results = None
best_model = None
best_model_name = None
test_metrics = None

dataset_profile = None
selected_model_names = None
model_reasoning = None

In [ ]:
def detect_task_type(y):
    if isinstance(y, pd.Series):
        y = y.to_frame()

    if y.shape[1] > 1:
        unique_values = [y[col].nunique() for col in y.columns]

        if all(v == 2 for v in unique_values):
            return "multi_output_binary_classification"

        return "multi_output_classification"

    target = y.iloc[:, 0]

    if pd.api.types.is_numeric_dtype(target) and target.nunique() > 20:
        return "regression"

    if target.nunique() == 2:
        return "binary_classification"

    return "multiclass_classification"

def build_dataset_profile(df, target_columns):
    X = df.drop(columns=target_columns)
    y = df[target_columns]

    detected_task_type = detect_task_type(y)

    profile = {
        "rows": int(df.shape[0]),
        "features": int(X.shape[1]),
        "target_columns": target_columns,
        "task_type": detected_task_type,
        "numeric_features": int(len(X.select_dtypes(include=["int64", "float64"]).columns)),
        "categorical_features": int(len(X.select_dtypes(include=["object", "category", "bool"]).columns)),
        "missing_values_total": int(df.isna().sum().sum()),
        "target_balance": {}
    }

    for col in target_columns:
        profile["target_balance"][col] = y[col].value_counts(normalize=True).to_dict()

    return profile


def evaluate_model(y_true, y_pred):
    global task_type

    if task_type == "regression":
        return {
            "mse": mean_squared_error(y_true, y_pred),
            "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
            "mae": mean_absolute_error(y_true, y_pred),
            "r2": r2_score(y_true, y_pred)
        }

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

def get_preprocessor(X):
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns

    numeric_transformer = Pipeline(steps=[
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

def build_all_models(task_type):
    if task_type == "regression":
        return {
            "Linear Regression": LinearRegression(),
            "Ridge": Ridge(),
            "Decision Tree": DecisionTreeRegressor(),
            "Random Forest": RandomForestRegressor(),
            "LightGBM": LGBMRegressor()
        }

    if task_type == "multi_output_binary_classification":
        return {
            "Logistic Regression": MultiOutputClassifier(
                LogisticRegression(max_iter=1000, class_weight="balanced")
            ),
            "Decision Tree": MultiOutputClassifier(
                DecisionTreeClassifier(class_weight="balanced")
            ),
            "Random Forest": MultiOutputClassifier(
                RandomForestClassifier(class_weight="balanced")
            ),
            "LightGBM": MultiOutputClassifier(
                LGBMClassifier(
                    class_weight="balanced",
                    random_state=random_state,
                    verbose=-1
                )
            ),
            "LightGBM Chain": ClassifierChain(
                LGBMClassifier(
                    class_weight="balanced",
                    random_state=random_state,
                    verbose=-1
                ),
                order="random",
                random_state=random_state
            )
        }

    return {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Gradient Boosting": GradientBoostingClassifier()
    }

def get_models(task_type):
    global selected_model_names

    all_models = build_all_models(task_type)

    if not selected_model_names:
        return all_models

    filtered_models = {}

    for name in selected_model_names:
        if name in all_models:
            filtered_models[name] = all_models[name]

    if len(filtered_models) == 0:
        return all_models

    return filtered_models

def fit(df, target_columns):
    global task_type
    global results
    global best_model
    global best_model_name
    global test_metrics

    X = df.drop(columns=target_columns)
    y = df[target_columns]

    y_strat = y.astype(str).apply(lambda x: "_".join(x), axis=1)

    task_type = detect_task_type(y)

    X_train_val, X_test, y_train_val, y_test, strat_train_val, strat_test = train_test_split(
        X,
        y,
        y_strat,
        test_size=test_size,
        random_state=random_state,
        stratify=y_strat
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=val_size,
        random_state=random_state,
        stratify=strat_train_val
    )

    preprocessor = get_preprocessor(X)
    models = get_models(task_type)

    local_results = []
    trained_models = {}

    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        y_val_pred = pipeline.predict(X_val)
        metrics = evaluate_model(y_val, y_val_pred)

        metrics["model"] = name
        metrics["selection_split"] = "validation"

        local_results.append(metrics)
        trained_models[name] = pipeline

    results = pd.DataFrame(local_results)

    if task_type == "regression":
        best_idx = results["rmse"].idxmin()
    else:
        best_idx = results["f1_macro"].idxmax()

    best_model_name = results.loc[best_idx, "model"]
    best_model = trained_models[best_model_name]

    y_test_pred = best_model.predict(X_test)
    test_metrics = evaluate_model(y_test, y_test_pred)

    for metric_name, metric_value in test_metrics.items():
        results.loc[best_idx, "test_" + metric_name] = metric_value

    return results

In [ ]:
@tool
def analyze_dataset_profile(dataset_path: str, target_columns: list[str]) -> str:
    """
    Analyze dataset and return task type, feature information and target balance.
    """
    global dataset_profile
    global task_type

    df = pd.read_csv(dataset_path)

    y = df[target_columns]
    mask = ~(y == 0).all(axis=1)
    df = df[mask]

    dataset_profile = build_dataset_profile(df, target_columns)
    task_type = dataset_profile["task_type"]

    return str(dataset_profile)

@tool
def get_available_models() -> str:
    """
    Return available model names for the detected task type.
    """
    global task_type

    if task_type is None:
        return "Сначала нужно вызвать analyze_dataset_profile."

    available = list(build_all_models(task_type).keys())

    return (
        f"Определенный тип задачи: {task_type}\n"
        f"Доступные модели: {available}\n\n"
        "Выбери несколько моделей из этого списка и затем вызови select_models."
    )

@tool
def select_models(model_names: list[str], reasoning: str) -> str:
    """
    Save selected model names and reasoning.
    """
    global selected_model_names
    global model_reasoning
    global task_type

    if task_type is None:
        return "Сначала нужно вызвать analyze_dataset_profile."

    allowed_models = set(build_all_models(task_type).keys())

    selected_model_names = [
        name for name in model_names
        if name in allowed_models
    ]

    model_reasoning = reasoning

    if len(selected_model_names) == 0:
        selected_model_names = list(allowed_models)
        model_reasoning = (
            reasoning
            + "\n\nНи одна из выбранных моделей не совпала со списком доступных, поэтому будут использованы все доступные модели."
        )

    return (
        f"Выбранные модели: {selected_model_names}\n\n"
        f"Обоснование выбора:\n{model_reasoning}"
    )

@tool
def train_selected_models(dataset_path: str, target_columns: list[str]) -> str:
    """
    Train selected ML models on train split, compare on validation split, and test the best model.
    """
    global model_reasoning

    df = pd.read_csv(dataset_path)
    # df = preprocess_dataset(df)

    y = df[target_columns]
    mask = ~(y == 0).all(axis=1)
    df = df[mask]

    final_results = fit(df, target_columns)

    return (
        "Обучение завершено.\n\n"
        f"Тип задачи: {task_type}\n\n"
        "Выбранные модели:\n"
        f"{selected_model_names}\n\n"
        "Обоснование выбора моделей:\n"
        f"{model_reasoning}\n\n"
        "Leaderboard на validation-выборке:\n"
        f"{final_results.to_string(index=False)}\n\n"
        f"Лучшая модель по validation f1_macro: {best_model_name}\n\n"
        "Финальные метрики лучшей модели на test-выборке:\n"
        f"{test_metrics}"
    )

@tool
def get_best_model() -> str:
    """
    Return the best model selected by the ML agent.
    """
    return f"Лучшая модель: {best_model_name}"


@tool
def get_task_type() -> str:
    """
    Return detected ML task type.
    """
    return f"Определенный тип задачи: {task_type}"


@tool
def get_model_reasoning() -> str:
    """
    Return reasoning for selected models and metrics.
    """
    return model_reasoning

llm_ml = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/nemotron-3-super-120b-a12b:free",

    temperature=0.2,
    top_p=0.9,
    max_tokens=2000
)

agent_ml = create_agent(
    model=llm_ml,
    tools=[
        analyze_dataset_profile,
        get_available_models,
        select_models,
        train_selected_models,
        get_best_model,
        get_task_type,
        get_model_reasoning
    ],
    system_prompt="""
Ты AutoML-агент.

Твой порядок действий обязателен:
1. Сначала вызови analyze_dataset_profile.
2. Затем вызови get_available_models.
3. На основе профиля данных сам выбери 2-4 наиболее релевантных моделей из списка доступных.
4. Затем вызови select_models и передай туда:
   - список выбранных моделей;
   - подробное обоснование выбора на русском языке.
5. Затем вызови train_selected_models.
6. После обучения объясни:
   - какой тип задачи был определен;
   - какие модели были выбраны и почему;
   - как были разделены данные;
   - какая модель победила на validation;
   - какие test-метрики получила лучшая модель.

Данные должны делиться на train, validation и test.
Модели обучаются только на train.
Выбор лучшей модели выполняется по validation.
Финальная проверка лучшей модели выполняется на test.

Не выдумывай метрики. Используй только значения, возвращенные инструментами.
"""
)

In [ ]:
response = agent_ml.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Обучи модели на cleaned_dataset.csv.

Целевые колонки:
Good for Party,
Good for Work/Study,
Good for Relaxation/Meditation,
Good for Driving.

Сначала проанализируй данные, затем сам выбери подходящие модели,
объясни выбор, обучи их и покажи leaderboard.
"""
        }
    ]
})

print(response["messages"][-1].content)

**Результаты AutoML‑потока для файла `cleaned_dataset.csv`**

---

### 1. Определённый тип задачи
- **multi_output_binary_classification** (множественная бинарная классификация)  
  – четыре целевых колонки: `Good for Party`, `Good for Work/Study`, `Good for Relaxation/Meditation`, `Good for Driving`.

### 2. Выбранные модели и обоснование
| Модель | Почему выбрана |
|--------|----------------|
| **Logistic Regression** | Простая линейная модель – служит базой для оценки линейной разделимости признаков и сложности задачи. |
| **Random Forest** | Ансамбль деревьев, устойчив к переобучению, хорошо работает со смешанными числовыми и категориальными признаками, не требует масштабирования. |
| **LightGBM Chain** | Градиентный бустинг, адаптированный для multi‑output через цепочку моделей, позволяет учитывать зависимости между метками (полезно при коррелированных целях). |

Эти три модели покрывают разный уровень сложности и подходы, что позволяет сравнить их эффективность на данном наборе д

In [ ]:
import json
import math
import joblib
from pathlib import Path
from datetime import datetime
from langchain.agents import create_agent
from langchain.tools import tool


REPORTS_DIR = Path("pipeline_reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_HISTORY_PATH = Path("run_history.json")
MODELS_DIR = Path("saved_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

stage_results = {}


def save_stage_result(stage_name: str, payload: dict):
    """Сохраняет результат этапа в json и txt файл."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    json_path = REPORTS_DIR / f"{stage_name}_{timestamp}.json"
    txt_path = REPORTS_DIR / f"{stage_name}_{timestamp}.txt"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"Этап: {stage_name}\n")
        f.write(f"Время: {timestamp}\n\n")

        if "input_file" in payload:
            f.write(f"Входной файл: {payload['input_file']}\n")
        if "output_file" in payload:
            f.write(f"Выходной файл: {payload['output_file']}\n")

        if "final_message" in payload:
            f.write("\nСообщение агента:\n")
            f.write(str(payload["final_message"]))
            f.write("\n\n")

        f.write("Структурированные данные:\n")
        f.write(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

    return {
        "json_path": str(json_path),
        "txt_path": str(txt_path)
    }


def sanitize_for_json(value):
    """Приводит значения к json-совместимому виду."""
    if isinstance(value, dict):
        return {k: sanitize_for_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_for_json(v) for v in value]
    if isinstance(value, tuple):
        return [sanitize_for_json(v) for v in value]
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value


def load_run_history():
    """Загружает историю запусков."""
    if RUN_HISTORY_PATH.exists():
        with open(RUN_HISTORY_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_run_history(history):
    """Сохраняет историю запусков."""
    with open(RUN_HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2, default=str)


def save_model_artifact(model_pipeline, run_id):
    """Сохраняет лучшую модель в joblib."""
    model_path = MODELS_DIR / f"best_model_{run_id}.joblib"
    joblib.dump(model_pipeline, model_path)
    return str(model_path)


def extract_model_details(model_pipeline):
    """
    Пытается достать параметры, веса или важности признаков из модели.
    Поддерживает Pipeline, MultiOutputClassifier и обычные sklearn-модели.
    """
    if model_pipeline is None:
        return {
            "model_params": None,
            "weights": None,
            "feature_importances": None
        }

    try:
        model = model_pipeline.named_steps["model"]
    except Exception:
        model = model_pipeline

    model_params = sanitize_for_json(model.get_params()) if hasattr(model, "get_params") else None

    feature_names = None
    try:
        if hasattr(model_pipeline, "named_steps") and "preprocessor" in model_pipeline.named_steps:
            preprocessor = model_pipeline.named_steps["preprocessor"]
            if hasattr(preprocessor, "get_feature_names_out"):
                feature_names = preprocessor.get_feature_names_out().tolist()
    except Exception:
        feature_names = None

    weights = None
    feature_importances = None

    try:
        if hasattr(model, "coef_"):
            coef = model.coef_
            if hasattr(coef, "tolist"):
                coef = coef.tolist()
            weights = {
                "type": "coef",
                "feature_names": feature_names,
                "values": sanitize_for_json(coef)
            }
        elif hasattr(model, "estimators_"):
            estimators_weights = []
            for idx, estimator in enumerate(model.estimators_):
                if hasattr(estimator, "coef_"):
                    coef = estimator.coef_
                    if hasattr(coef, "tolist"):
                        coef = coef.tolist()
                    estimators_weights.append({
                        "estimator_index": idx,
                        "feature_names": feature_names,
                        "values": sanitize_for_json(coef)
                    })
            if estimators_weights:
                weights = {
                    "type": "multi_estimator_coef",
                    "estimators": estimators_weights
                }
    except Exception:
        weights = None

    try:
        if hasattr(model, "feature_importances_"):
            values = model.feature_importances_
            if hasattr(values, "tolist"):
                values = values.tolist()
            feature_importances = {
                "type": "feature_importances",
                "feature_names": feature_names,
                "values": sanitize_for_json(values)
            }
        elif hasattr(model, "estimators_"):
            estimators_importances = []
            for idx, estimator in enumerate(model.estimators_):
                if hasattr(estimator, "feature_importances_"):
                    values = estimator.feature_importances_
                    if hasattr(values, "tolist"):
                        values = values.tolist()
                    estimators_importances.append({
                        "estimator_index": idx,
                        "feature_names": feature_names,
                        "values": sanitize_for_json(values)
                    })
            if estimators_importances:
                feature_importances = {
                    "type": "multi_estimator_feature_importances",
                    "estimators": estimators_importances
                }
    except Exception:
        feature_importances = None

    return {
        "model_params": model_params,
        "weights": weights,
        "feature_importances": feature_importances
    }


def compare_runs(current_run, previous_run, metric_name="f1_macro", tolerance=0.01):
    """Сравнивает текущий запуск с предыдущим по выбранной метрике."""
    current_metrics = current_run.get("test_metrics", {}) or {}
    previous_metrics = previous_run.get("test_metrics", {}) or {}

    current_value = current_metrics.get(metric_name)
    previous_value = previous_metrics.get(metric_name)

    if current_value is None or previous_value is None:
        return {
            "status": "not_available",
            "message": f"Не удалось сравнить по метрике {metric_name}."
        }

    diff = current_value - previous_value

    if abs(diff) < tolerance:
        verdict = "примерно так же"
    elif diff > 0:
        verdict = "лучше"
    else:
        verdict = "хуже"

    return {
        "status": "ok",
        "metric": metric_name,
        "previous_value": previous_value,
        "current_value": current_value,
        "difference": diff,
        "verdict": verdict,
        "message": (
            f"Сравнение с предыдущим запуском по {metric_name}: "
            f"было {previous_value:.4f}, стало {current_value:.4f}, "
            f"разница {diff:+.4f}. Итог: стало {verdict}."
        )
    }


def persist_ml_run(dataset_path: str):
    """Сохраняет ML-запуск, модель и результаты в историю."""
    global dataset_profile
    global task_type
    global best_model
    global best_model_name
    global selected_model_names
    global test_metrics
    global results

    run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    model_path = save_model_artifact(best_model, run_id) if best_model is not None else None
    model_details = extract_model_details(best_model)

    current_run = {
        "run_id": run_id,
        "dataset_path": dataset_path,
        "rows": dataset_profile["rows"] if dataset_profile else None,
        "features": dataset_profile["features"] if dataset_profile else None,
        "task_type": task_type,
        "best_model_name": best_model_name,
        "selected_models": sanitize_for_json(selected_model_names),
        "test_metrics": sanitize_for_json(test_metrics),
        "leaderboard": sanitize_for_json(results.to_dict(orient="records")) if results is not None else None,
        "model_artifact_path": model_path,
        "model_params": model_details["model_params"],
        "weights": model_details["weights"],
        "feature_importances": model_details["feature_importances"]
    }

    history = load_run_history()
    previous_run = history[-1] if history else None

    comparison = compare_runs(current_run, previous_run) if previous_run else {
        "status": "first_run",
        "message": "Это первый запуск, сравнение пока недоступно."
    }

    history.append(current_run)
    save_run_history(history)

    return current_run, comparison


def save_final_pipeline_report(summary: dict):
    """Сохраняет общий итоговый txt-отчет по всему пайплайну."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    report_path = REPORTS_DIR / f"00_final_pipeline_report_{timestamp}.txt"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("ИТОГОВЫЙ ОТЧЕТ ПО ML-ПАЙПЛАЙНУ\n")
        f.write(f"Время: {timestamp}\n\n")

        if "download" in summary:
            f.write("1. Загрузка и подготовка\n")
            f.write(f"Статус: {summary['download'].get('status')}\n")
            f.write(f"Выходной файл: {summary['download'].get('output_file')}\n")
            f.write(f"Отчет: {summary['download'].get('saved_files', {}).get('txt_path')}\n")
            f.write("Проверки:\n")
            f.write(json.dumps(summary["download"].get("checks", {}), ensure_ascii=False, indent=2))
            f.write("\n\n")
            f.write(str(summary["download"].get("final_message", "")))
            f.write("\n\n")

        if "eda" in summary:
            f.write("2. EDA\n")
            f.write(f"Статус: {summary['eda'].get('status')}\n")
            f.write(f"Входной файл: {summary['eda'].get('input_file')}\n")
            f.write(f"Отчет: {summary['eda'].get('saved_files', {}).get('txt_path')}\n\n")
            f.write(str(summary["eda"].get("final_message", "")))
            f.write("\n\n")

        if "markup" in summary:
            f.write("3. Разметка\n")
            f.write(f"Статус: {summary['markup'].get('status')}\n")
            f.write(f"Входной файл: {summary['markup'].get('input_file')}\n")
            f.write(f"Выходной файл: {summary['markup'].get('output_file')}\n")
            f.write(f"Отчет: {summary['markup'].get('saved_files', {}).get('txt_path')}\n")
            f.write("Проверки:\n")
            f.write(json.dumps(summary["markup"].get("checks", {}), ensure_ascii=False, indent=2))
            f.write("\n\n")
            f.write(str(summary["markup"].get("final_message", "")))
            f.write("\n\n")

        if "preproc" in summary:
            f.write("4. Препроцессинг\n")
            f.write(f"Статус: {summary['preproc'].get('status')}\n")
            f.write(f"Входной файл: {summary['preproc'].get('input_file')}\n")
            f.write(f"Выходной файл: {summary['preproc'].get('output_file')}\n")
            f.write(f"Отчет: {summary['preproc'].get('saved_files', {}).get('txt_path')}\n")
            f.write("Проверки:\n")
            f.write(json.dumps(summary["preproc"].get("checks", {}), ensure_ascii=False, indent=2))
            f.write("\n\n")
            f.write(str(summary["preproc"].get("final_message", "")))
            f.write("\n\n")

        if "ml" in summary:
            f.write("5. ML\n")
            f.write(f"Статус: {summary['ml'].get('status')}\n")
            f.write(f"Входной файл: {summary['ml'].get('input_file')}\n")
            f.write(f"Отчет: {summary['ml'].get('saved_files', {}).get('txt_path')}\n")
            f.write(f"Лучшая модель: {summary['ml'].get('artifacts', {}).get('best_model_name')}\n")
            f.write(f"Тип задачи: {summary['ml'].get('artifacts', {}).get('task_type')}\n")
            f.write(f"Файл модели: {summary['ml'].get('artifacts', {}).get('model_artifact_path')}\n")
            f.write("Test metrics:\n")
            f.write(json.dumps(summary["ml"].get("artifacts", {}).get("test_metrics", {}), ensure_ascii=False, indent=2))
            f.write("\n\n")

            comparison = summary["ml"].get("artifacts", {}).get("comparison_with_previous")
            if comparison:
                f.write("Сравнение с предыдущим запуском:\n")
                f.write(json.dumps(comparison, ensure_ascii=False, indent=2))
                f.write("\n\n")

            f.write(str(summary["ml"].get("final_message", "")))
            f.write("\n\n")

        if "pipeline_state" in summary:
            f.write("6. Итоговое состояние пайплайна\n")
            f.write(f"Отчет: {summary['pipeline_state'].get('saved_files', {}).get('txt_path')}\n")
            f.write(json.dumps(summary["pipeline_state"], ensure_ascii=False, indent=2, default=str))
            f.write("\n")

    return str(report_path)


@tool
def run_download_agent() -> str:
    """Запускает агент загрузки и подготовки. Должен создать spotify_original.csv и spotify_clean.csv."""
    global agent_state

    try:
        agent_state = {
            "df_raw": None,
            "df_clean": None,
            "preprocessing_log": []
        }

        response = agent_download.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Выполни полный пайплайн подготовки данных по всем 13 шагам. "
                            "После каждого шага кратко объясняй, что сделал и почему. "
                            "Сначала сохрани исходный отфильтрованный файл как spotify_original.csv, "
                            "а в конце сохрани очищенный файл как spotify_clean.csv."
                        )
                    }
                ]
            },
            config={"recursion_limit": 80}
        )

        final_message = str(response["messages"][-1].content).strip()

        result = {
            "agent": "agent_download",
            "status": "ok" if Path("spotify_clean.csv").exists() and Path("spotify_original.csv").exists() else "error",
            "output_file": "spotify_clean.csv",
            "final_message": final_message,
            "checks": {
                "spotify_original_exists": Path("spotify_original.csv").exists(),
                "spotify_clean_exists": Path("spotify_clean.csv").exists()
            }
        }
    except Exception as e:
        result = {
            "agent": "agent_download",
            "status": "error",
            "output_file": "spotify_clean.csv",
            "final_message": f"Ошибка: {type(e).__name__}: {e}"
        }

    saved = save_stage_result("01_download", result)
    result["saved_files"] = saved
    stage_results["download"] = result

    return json.dumps(result, ensure_ascii=False, default=str)


@tool
def run_eda_agent(input_path: str = "spotify_original.csv") -> str:
    """Запускает EDA-агента по файлу spotify_original.csv."""
    try:
        response = agent_eda.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": f"""
Сделай EDA-анализ датасета по пути: {input_path}.

Нужно:
1. Посмотреть структуру данных.
2. Дать краткое описание датасета.
3. Указать возможные проблемы.
4. Предложить идеи для ML.
"""
                }
            ]
        })

        final_message = str(response["messages"][-1].content).strip()

        result = {
            "agent": "agent_eda",
            "status": "ok" if final_message else "error",
            "input_file": input_path,
            "final_message": final_message
        }
    except Exception as e:
        result = {
            "agent": "agent_eda",
            "status": "error",
            "input_file": input_path,
            "final_message": f"Ошибка: {type(e).__name__}: {e}"
        }

    saved = save_stage_result("02_eda", result)
    result["saved_files"] = saved
    stage_results["eda"] = result

    return json.dumps(result, ensure_ascii=False, default=str)


@tool
def run_markup_agent(input_path: str = "spotify_clean.csv") -> str:
    """Запускает агента разметки по файлу spotify_clean.csv и создает df_final.csv."""
    global feature_selection
    feature_selection = None

    output_file = "df_final.csv"

    try:
        response = agent_razm.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": f"""
У тебя есть исходный датасет по пути: {input_path}.

Нужно:
1. Подготовить датасет для агента: убрать колонки Good for..., потому что по ним нельзя классифицировать.
2. Сохранить подготовленный файл рядом с исходным датасетом.
3. Выбрать признаки для классификации.
4. Классифицировать песни по 4 плейлистам: party, work, relax, drive.
5. Сохранить результат в classified_songs.csv.
6. Вернуть в финальный датасет колонки Good for из исходного файла.
7. Сохранить финальный файл как {output_file}.
"""
                }
            ]
        })

        result = {
            "agent": "agent_razm",
            "status": "ok" if Path(output_file).exists() else "error",
            "input_file": input_path,
            "output_file": output_file,
            "final_message": str(response["messages"][-1].content),
            "checks": {
                "spotify_for_agent_exists": Path("spotify_for_agent.csv").exists(),
                "classified_songs_exists": Path("classified_songs.csv").exists(),
                "df_final_exists": Path(output_file).exists()
            }
        }
    except Exception as e:
        result = {
            "agent": "agent_razm",
            "status": "error",
            "input_file": input_path,
            "output_file": output_file,
            "final_message": f"Ошибка: {type(e).__name__}: {e}"
        }

    saved = save_stage_result("03_markup", result)
    result["saved_files"] = saved
    stage_results["markup"] = result

    return json.dumps(result, ensure_ascii=False, default=str)


@tool
def run_preproc_agent(input_path: str = "df_final.csv") -> str:
    """Запускает агента препроцессинга по df_final.csv и создает cleaned_dataset.csv."""
    output_file = "cleaned_dataset.csv"

    try:
        response = agent_preproc.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": f"""
Подготовь датасет {input_path} для ML-задачи.

Нужно:
1. Проанализировать колонки.
2. Решить, какие object-колонки удалить, а какие закодировать через one-hot encoding.
3. Сохранить итоговый файл как {output_file}.
"""
                }
            ]
        })

        result = {
            "agent": "agent_preproc",
            "status": "ok" if Path(output_file).exists() else "error",
            "input_file": input_path,
            "output_file": output_file,
            "final_message": str(response["messages"][-1].content),
            "checks": {
                "analysis_exists": Path("analysis.json").exists(),
                "decision_exists": Path("decision.json").exists(),
                "cleaned_dataset_exists": Path(output_file).exists()
            }
        }
    except Exception as e:
        result = {
            "agent": "agent_preproc",
            "status": "error",
            "input_file": input_path,
            "output_file": output_file,
            "final_message": f"Ошибка: {type(e).__name__}: {e}"
        }

    saved = save_stage_result("04_preproc", result)
    result["saved_files"] = saved
    stage_results["preproc"] = result

    return json.dumps(result, ensure_ascii=False, default=str)


@tool
def run_ml_agent(dataset_path: str = "cleaned_dataset.csv") -> str:
    """Запускает ML-агента по cleaned_dataset.csv, сохраняет модель и историю запусков."""
    global task_type
    global results
    global best_model
    global best_model_name
    global test_metrics
    global dataset_profile
    global selected_model_names
    global model_reasoning

    task_type = None
    results = None
    best_model = None
    best_model_name = None
    test_metrics = None
    dataset_profile = None
    selected_model_names = []
    model_reasoning = ""

    target_columns = [
        "Good for Party",
        "Good for Work/Study",
        "Good for Relaxation/Meditation",
        "Good for Driving"
    ]

    try:
        response = agent_ml.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": f"""
Обучи модели на датасете {dataset_path}.

Целевые колонки:
{", ".join(target_columns)}

Сначала проанализируй данные, затем сам выбери подходящие модели,
объясни выбор, обучи их и покажи leaderboard.
"""
                }
            ]
        })

        current_run, comparison = persist_ml_run(dataset_path)

        result = {
            "agent": "agent_ml",
            "status": "ok" if best_model is not None else "error",
            "input_file": dataset_path,
            "final_message": str(response["messages"][-1].content),
            "checks": {
                "task_type_detected": task_type is not None,
                "best_model_exists": best_model is not None,
                "best_model_name_exists": best_model_name is not None,
                "test_metrics_exists": test_metrics is not None
            },
            "artifacts": {
                "task_type": task_type,
                "selected_models": sanitize_for_json(selected_model_names),
                "best_model_name": best_model_name,
                "test_metrics": sanitize_for_json(test_metrics),
                "leaderboard": sanitize_for_json(results.to_dict(orient="records")) if results is not None else None,
                "dataset_profile": sanitize_for_json(dataset_profile),
                "model_reasoning": model_reasoning,
                "model_artifact_path": current_run.get("model_artifact_path"),
                "model_params": current_run.get("model_params"),
                "weights": current_run.get("weights"),
                "feature_importances": current_run.get("feature_importances"),
                "comparison_with_previous": comparison
            }
        }
    except Exception as e:
        result = {
            "agent": "agent_ml",
            "status": "error",
            "input_file": dataset_path,
            "final_message": f"Ошибка: {type(e).__name__}: {e}"
        }

    saved = save_stage_result("05_ml", result)
    result["saved_files"] = saved
    stage_results["ml"] = result

    return json.dumps(result, ensure_ascii=False, default=str)


@tool
def check_pipeline_state() -> str:
    """Проверяет, какие файлы и ML-артефакты созданы, и сохраняет итоговый отчет."""
    state = {
        "agent": "pipeline_state_checker",
        "status": "ok",
        "files": {
            "spotify_original.csv": Path("spotify_original.csv").exists(),
            "spotify_clean.csv": Path("spotify_clean.csv").exists(),
            "spotify_for_agent.csv": Path("spotify_for_agent.csv").exists(),
            "classified_songs.csv": Path("classified_songs.csv").exists(),
            "df_final.csv": Path("df_final.csv").exists(),
            "analysis.json": Path("analysis.json").exists(),
            "decision.json": Path("decision.json").exists(),
            "cleaned_dataset.csv": Path("cleaned_dataset.csv").exists(),
            "run_history.json": RUN_HISTORY_PATH.exists()
        },
        "ml_state": {
            "task_type": globals().get("task_type"),
            "best_model_name": globals().get("best_model_name"),
            "best_model_exists": globals().get("best_model") is not None,
            "test_metrics": globals().get("test_metrics")
        }
    }

    saved = save_stage_result("06_pipeline_state", state)
    state["saved_files"] = saved
    stage_results["pipeline_state"] = state

    return json.dumps(state, ensure_ascii=False, default=str)


@tool
def build_final_pipeline_report() -> str:
    """Собирает единый итоговый txt-отчет по сохраненным результатам этапов."""
    if not stage_results:
        return json.dumps({
            "status": "error",
            "message": "Нет сохраненных результатов этапов для сборки итогового отчета."
        }, ensure_ascii=False)

    report_path = save_final_pipeline_report(stage_results)

    result = {
        "status": "ok",
        "final_report_path": report_path
    }

    return json.dumps(result, ensure_ascii=False, default=str)


orchestrator_tools = [
    run_download_agent,
    run_eda_agent,
    run_markup_agent,
    run_preproc_agent,
    run_ml_agent,
    check_pipeline_state,
    build_final_pipeline_report
]

agent_orchestrator = create_agent(
    model=llm_eda,
    tools=orchestrator_tools,
    system_prompt="""
Ты агент-оркестратор полного ML-пайплайна.

Порядок действий обязателен:
1. Сначала вызови run_download_agent.
2. Если этап успешен, вызови run_eda_agent и передай файл spotify_original.csv.
3. Если этап успешен, вызови run_markup_agent и передай файл spotify_clean.csv.
4. Если этап успешен, вызови run_preproc_agent и передай файл df_final.csv.
5. Если этап успешен, вызови run_ml_agent и передай файл cleaned_dataset.csv.
6. В конце вызови check_pipeline_state.
7. После этого вызови build_final_pipeline_report.
8. После каждого этапа опирайся только на данные из tool output.
9. Если какой-то этап вернул status=error, остановись.

Критично важно:
- EDA должен работать по spotify_original.csv.
- Все последующие этапы после загрузки должны использовать результат загрузки, то есть spotify_clean.csv, а затем файлы следующих этапов.
- Не придумывай имена файлов.
- Не пропускай этапы.

В финальном ответе сообщи:
- статус каждого этапа,
- какие файлы были входом и выходом,
- где сохранены отчеты,
- где лежит общий итоговый txt-файл,
- какая модель лучшая,
- какие test-метрики получены,
- стало лучше, хуже или примерно так же по сравнению с предыдущим запуском.

Пиши по-русски.
"""
)

In [ ]:
response = agent_orchestrator.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Запусти полный ML-пайплайн.

Порядок обязателен:
1. Сначала выполни загрузку и подготовку данных.
2. Затем сделай EDA по файлу spotify_original.csv.
3. Затем выполни разметку по файлу spotify_clean.csv.
4. Затем выполни препроцессинг по файлу df_final.csv.
5. Затем обучи модели по файлу cleaned_dataset.csv.
6. Сохрани отчеты каждого этапа.
7. Собери общий итоговый txt-отчет.
8. Сохрани результат запуска в долговременную историю.
9. В конце сравни текущий запуск с предыдущим и скажи, стало лучше, хуже или примерно так же.

Не пропускай этапы и используй только указанные файлы.
"""
        }
    ]
})

print(response["messages"][-1].content)


Скачиваю: devdope/900k-spotify...
Good for фильтр: 551,443 → 107,411 строк
Ограничено до 100k строк
финальный отчет
Было:  100,000 строк × 25 колонок
Стало: 85,593 строк × 38 колонок

Шаги предобработки:
  1. Дубликаты: удалено 14403 полных + 0 по Artist+song
  2. song: 3 (0.0%) → мода 'Home'
  3. Album: 4 (0.0%) → мода 'L.A.B IV'
  4. Time signature: 3 (0.0%) → мода '4/4'
  5. Length → length_seconds (секунды)
  6. Release Date → release_year + release_month
  7. Loudness (db) → loudness_db (float)
  8. Explicit → explicit_int (0/1)
  9. Key → key_note + key_mode (0=min, 1=Maj)
  10. Time signature → time_sig (int)
  11. emotion: оставлены только ['joy', 'sadness', 'anger', 'fear', 'love', 'surprise'], удалено 4 мусорных строк
  12. Genre → 7 бинарных столбцов жанров
  13. Tempo: 632 выбросов → clip [31.50, 179.50]
  14. Danceability: 2,007 выбросов → clip [20.50, 104.50]
  15. Speechiness: 11,870 выбросов → clip [-6.00, 18.00]
  16. Liveness: 7,615 выбросов → clip [-5.00, 35.00]
  17

Batches: 100%|████████████████████████████████| 367/367 [21:18<00:00,  3.48s/it]


✅ **Этап 7 успешен!**
- Итоговый отчёт собран
- Путь: `pipeline_reports/00_final_pipeline_report_2026-04-28_03-16-53.txt`

---

## 📋 ИТОГОВЫЙ РЕЗУЛЬТАТ ПОЛНОГО ML-ПАЙПЛАЙНА

### ✅ Статус каждого этапа:

| Этап | Статус | Входной файл | Выходной файл | Отчёт |
|------|--------|--------------|---------------|-------|
| 1️⃣ Загрузка & Подготовка | ✅ OK | - | `spotify_original.csv`, `spotify_clean.csv` | `01_download_2026-04-28_02-52-28.txt` |
| 2️⃣ EDA | ✅ OK | `spotify_original.csv` (100K × 25) | Анализ | `02_eda_2026-04-28_02-52-51.txt` |
| 3️⃣ Разметка | ✅ OK | `spotify_clean.csv` | `df_final.csv` (5.5K × признаки) | `03_markup_2026-04-28_03-14-43.txt` |
| 4️⃣ Препроцессинг | ✅ OK | `df_final.csv` | `cleaned_dataset.csv` (5.5K × 49) | `04_preproc_2026-04-28_03-14-53.txt` |
| 5️⃣ Обучение ML | ✅ OK | `cleaned_dataset.csv` | Модель + метрики | `05_ml_2026-04-28_03-16-47.txt` |
| 6️⃣ Проверка состояния | ✅ OK | - | Отчёт | `06_pipeline_state_2026-04-28_03-16-51.txt` |
| 7️⃣ Финальный отчё